# High-Z Filterbank: Exploring the Excess Noise

## Goal

We have observed excess variation/noise in the High-Z filterbank data. It initially appeared to be mainly a high-frequency problem, but it now seems to occur across much of the observing band and may be worse for some physical filter channels than others.

The goal today is **not to determine the cause of the noise**. Instead, we want to explore data from the Adak deployment and look for patterns that can guide later hardware tests.

Questions to keep in mind:

- Are some channels consistently noisier than others?
- Does the behavior depend on LO frequency?
- Does it change with time?
- Do several channels show unusual behavior at the same time?
- Are there spikes, jumps, or slow drifts?

Write down anything unusual you notice, even if you cannot explain it yet.

## Python setup

You will need Python and a few common scientific Python packages for this notebook.

### Option 1: Using `venv`

Open a terminal in the folder where you want to work and create a virtual environment:

```bash
python3 -m venv highz-env
```

Activate it.

**macOS / Linux:**

```bash
source highz-env/bin/activate
```

**Windows:**

```bash
highz-env\Scripts\activate
```

Then install the packages used in this notebook:

```bash
python -m pip install numpy matplotlib astropy jupyter
```

Start Jupyter with:

```bash
jupyter notebook
```

Open this notebook and make sure `state_0.fits` is in the **same folder as the notebook**.

### Option 2: If you already use Conda

```bash
conda create -n highz-filterbank python=3.12 numpy matplotlib astropy jupyter
conda activate highz-filterbank
jupyter notebook
```

### Quick check

Run the next cell. If it completes without an error, the required packages are installed correctly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

print("Python environment is ready.")


## 1. Filterbank basics

A spectrometer measures how much RF power is present at different frequencies.

The High-Z receiver feeds two different spectrometers:

```text
                         ┌── RFSoC spectrometer
Antenna → Receiver ──────┤
                         └── Filterbank spectrometer
```

The analog filterbank contains **21 cavity-filter channels**:

```text
                       ┌── Filter 1  ── Detector 1
                       ├── Filter 2  ── Detector 2
RF signal → split ─────┼── Filter 3  ── Detector 3
                       │       ...
                       └── Filter 21 ── Detector 21
```

The filters operate near 900 MHz. A mixer and **local oscillator (LO)** translate different parts of the incoming RF spectrum into the frequency range of these fixed filters.

The following operation is defined as one **sweep**:

```text
LO setting 1 → measure all 21 filters
LO setting 2 → measure all 21 filters
...
LO setting 144 → measure all 21 filters
```

This distinction is useful for troubleshooting: unusual behavior might follow a **physical filter channel**, or it might appear only at particular **LO/frequency settings**.

## 2. Load the data

Start with `state_0.fits`. In this Adak observing cycle, this state contains **225 repeated sweeps**.

Each sweep contains:

- 144 LO settings
- 21 filter channels
- 144 × 21 = 3024 measurements

We will first inspect the file rather than assuming that we understand it.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

# Change this if the FITS file is stored somewhere else.
filename = Path("state_0.fits")

hdul = fits.open(filename)
hdul.info()

### Inspect the header and table

Look through the output below. You do not need to understand every FITS keyword.

Useful things to identify are the number of filters, LO points, and spectra.

In [ ]:
print(hdul[0].header)

In [ ]:
data = hdul[1].data

print("Columns:")
print(data.columns.names)

print("\nNumber of table rows:", len(data))

Now inspect the main columns used in the analysis.

In [ ]:
for name in ["DATA_CUBE", "LO_FREQUENCIES", "SPECTRUM_INDEX", "SPECTRUM_TIMESTAMP"]:
    arr = data[name]
    print(f"{name:20s} shape = {arr.shape}, dtype = {arr.dtype}")

## 3. Reshape the measurements

The acquisition program stores the 21 filter measurements for each LO setting consecutively.

One sweep can therefore be reshaped as:

```text
(144 LO settings, 21 filters)
```

and all 225 repeated sweeps can be organized as:

```text
(sweep, LO setting, filter)
```

or, for this file:

```text
(225, 144, 21)
```

In [ ]:
n_sweeps = len(data)
n_lo = int(hdul[0].header["N_LO_PTS"])
n_filters = int(hdul[0].header["N_FILTERS"])

cube = np.asarray(data["DATA_CUBE"]).reshape(n_sweeps, n_lo, n_filters)
lo_freq = np.asarray(data["LO_FREQUENCIES"][0])

print("Data cube shape:", cube.shape)
print("LO frequency shape:", lo_freq.shape)
print("LO range:", lo_freq.min(), "to", lo_freq.max(), "MHz")

## 4. Look at one repeated measurement

Start with one filter and one LO setting.

The plot below shows how that exact measurement changes from sweep to sweep. Try changing `filter_number` and `lo_index`.

Look for:

- ordinary scatter
- large spikes
- sudden jumps
- slow drift
- periodic behavior

In [ ]:
filter_number = 1   # 1 through 21
lo_index = 0        # 0 through 143

values = cube[:, lo_index, filter_number - 1]

plt.figure(figsize=(9, 4.5))
plt.plot(np.arange(n_sweeps), values, marker=".", linestyle="-")
plt.xlabel("Sweep number")
plt.ylabel("Raw measurement")
plt.title(
    f"Filter {filter_number}, LO = {lo_freq[lo_index]:.1f} MHz"
)
plt.grid(alpha=0.25)
plt.show()

### Try a few channels

Change the filter number and LO index above.

**Notes:**  
Which combinations look stable? Which look unusual? Are there obvious spikes or drifts?

## 5. Compare the 21 filters

A simple way to characterize the variation is to calculate the standard deviation across the repeated sweeps.

For each LO setting and filter, we calculate how much the 225 measurements vary.

This does **not** prove that the variation is electronic noise. At this stage we are simply using it as a way to identify unusual channels.

In [ ]:
# Standard deviation over the repeated-sweep axis.
noise = np.std(cube.astype(float), axis=0, ddof=1)

print("Noise array shape:", noise.shape)
# (144 LO settings, 21 filters)

First, summarize each filter by taking the median standard deviation across the LO sweep.

In [ ]:
noise_by_filter = np.median(noise, axis=0)

plt.figure(figsize=(9, 4.5))
plt.plot(np.arange(1, n_filters + 1), noise_by_filter, marker="o")
plt.xlabel("Filter number")
plt.ylabel("Median standard deviation")
plt.title("Variation by physical filter channel")
plt.xticks(np.arange(1, n_filters + 1))
plt.grid(alpha=0.25)
plt.show()

**Questions:**

- Are a few filters clearly worse than the others?
- Are neighboring filters similar?
- Are there any surprising outliers?

Write down the filter numbers that look most interesting.

## 6. Does the variation depend on LO frequency?

Choose a few filters from the previous plot — for example, one quiet channel and one noisy channel — and compare their variation across the LO sweep.

In [ ]:
filters_to_plot = [1, 2, 3]  # Change these after looking at the previous figure

plt.figure(figsize=(9, 4.5))

for f in filters_to_plot:
    plt.plot(lo_freq, noise[:, f - 1], marker=".", label=f"Filter {f}")

plt.xlabel("LO frequency (MHz)")
plt.ylabel("Standard deviation")
plt.title("Variation across the LO sweep")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

Ask:

> Does a noisy filter remain noisy across much of the LO sweep, or is it only noisy at particular LO settings?

If one physical filter is consistently worse over many LO settings, that may point toward hardware associated with that channel.

If many filters show unusual behavior at the same LO settings, that may point toward something shared elsewhere in the system.

## 7. Compare channels over time

If two filters show spikes or changes at the same time, that could be an important clue.

Choose one LO setting and a few interesting filters. The code below normalizes each trace so that channels with different average levels are easier to compare.

In [ ]:
lo_index = 0
filters_to_compare = [1, 2, 3]  # Replace with interesting channels

plt.figure(figsize=(9, 4.5))

for f in filters_to_compare:
    y = cube[:, lo_index, f - 1].astype(float)
    y_norm = (y - np.mean(y)) / np.std(y)
    plt.plot(np.arange(n_sweeps), y_norm, label=f"Filter {f}")

plt.xlabel("Sweep number")
plt.ylabel("Normalized variation")
plt.title(f"Channel comparison at LO = {lo_freq[lo_index]:.1f} MHz")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 8. Summarize findings

We are looking for differences between filters at one LO setting, or for a specific LO setting which triggers noisy reponses.

### Notes

**Filters that appear unusually noisy:**


**Does the behavior depend on LO frequency?**


**Any spikes, jumps, or drift?**


**Do multiple filters behave similarly at the same time?**


**Anything else unusual?**

## 9. If there is time

Once `state_0.fits` makes sense, the same basic analysis can be applied to some of the other `state_*.fits` files, which correspond to different input/calibration states.

A useful later question is:

> **Do the same physical filters remain noisy when the input state changes?**

We can go through what each state represents before interpreting those results.

---

## Why this helps

The purpose of this analysis is to decide what to test in the hardware.

For example, if one filter channel is consistently bad, we can later swap an amplifier from that channel with one from a quiet channel.

- If the problem **moves with the amplifier**, the amplifier becomes suspicious.
- If the problem **stays with the original channel**, we investigate another part of that signal path.

The same idea can be used with splitters, cables, detectors, and other components.

For now, focus on **finding and documenting patterns in the data** rather than determining their cause.